In [2]:
!apt-get update
!apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:5 https://cli.github.com/packages stable/main amd64 Packages [355 B]
Get:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:8 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [100 kB]
Get:9 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:10 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,820 kB]
Get:11 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:12 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [10.5 MB]
Get:13 http://archive.ubuntu.com/ubuntu jammy-updates/restricted a

In [3]:
!nohup ollama serve >/dev/null 2>&1 &

In [4]:
!ollama pull gemma3:4b

In [5]:
!pip install -q ollama

In [6]:
!pip install -q langchain-ollama

In [7]:
# YouTube
%pip install -U youtube-search youtube-transcript-api pytube

# LangChain
%pip install -U langchain
%pip install -U langchain-core
%pip install -U langchain-community
%pip install -U langchain-text-splitters
%pip install -U langchain-huggingface
%pip install -U langchain-chroma

# Embedding
%pip install -U sentence-transformers

# Vector DB
%pip install -U chromadb

# Reranker
%pip install -U flashrank

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 485.2/485.2 kB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.6/57.6 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 136.9/136.9 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 558.3/558.3 kB 27.3 MB/s eta 0:00:00
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.4.8
    Uninstalling langchain-core-1.4.8:
      Successfully uninstalled langchain-core-1.4.8
  Attempting uninstall: langchain
    Found existing installation: langchain 1.3.11
    Uninstalling langchain-1.3.11:
      Successfully uninstalled langchain-1.3.11
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 45.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 73.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 6.1 MB/s eta 0:00:00
  Attempting uninst

In [8]:
### 1. 환경 설정 하기
from youtube_search import YoutubeSearch
from langchain_community.document_loaders import YoutubeLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

/tmp/ipykernel_1241/3873709336.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import YoutubeLoader


In [16]:
### 2. Youtube 다운로드
videos = YoutubeSearch("미국 대선", max_results=5).to_dict()

docs = []
cnt = 0
for v in enumerate(videos, start=1 ):
  try:
    v["video_url"] = "https://youtube.com" + v["url_suffix"]
    loader = YoutubeLoader.from_youtube_url(
      v["video_url"],
      language=["ko", "en"]
    )
    print("비디오 자막")
  except Exception as e:
    print(e)
    continue
  docs.extend(loader.load())

tuple indices must be integers or slices, not str
tuple indices must be integers or slices, not str
tuple indices must be integers or slices, not str
tuple indices must be integers or slices, not str
tuple indices must be integers or slices, not str


In [15]:
### 3. Splitter
splitter = RecursiveCharacterTextSplitter(
    chunk_size=700,
    chunk_overlap=100
)

split_docs = splitter.split_documents(docs)
print(len(split_docs))

0


In [13]:
### 4. HuggingFace Embedding
embedding = HuggingFaceEmbeddings(
    model_name="BAAI/bge-m3",
    model_kwargs={
        "device":"cuda"
    },
    encode_kwargs={
        "normalize_embeddings":True
    }
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/444 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

In [16]:
### 5. VectorStore
vectorstore = Chroma.from_documents(
    split_docs,
    embedding
)

In [ ]:
### 6. retriever
base_retriever = vectorstore.as_retriever(
    search_kwargs={"k":10}
)

In [ ]:
### 7. FlashRank
from langchain_classic.retrievers.document_compressors import  FlashrankRerank
reranker = FlashrankRerank(top_n=3)

In [ ]:
### 8. redundant_filter
from langchain_community.document_transformers import EmbeddingsRedundantFilter
redundant_filter = EmbeddingsRedundantFilter(embeddings=embedding)

In [ ]:
### 9. LLM
model = ChatOllama(
    model="gemma3:4b",
    temperature=0.7
)

In [ ]:
### 10. Document Compressor
from langchain_classic.retrievers.document_compressors import LLMChainExtractor
llm_compressor = LLMChainExtractor.from_llm(model)

In [ ]:
### 11. Document Transformer
from langchain_community.document_transformers import LongContextReorder
reorder = LongContextReorder()

In [ ]:
### 12. Compressor Pipeline : 순서가 무척 중요하다.
from langchain_classic.retrievers.document_compressors import  DocumentCompressorPipeline
from langchain_classic.retrievers import ContextualCompressionRetriever
pipeline_compressor = DocumentCompressorPipeline(
    transformers=[
        redundant_filter,
        reranker,
        reorder,
        llm_compressor
    ]
)

In [ ]:
### 13. ContextualCompressionRetriever
post_retriever = ContextualCompressionRetriever(
    base_compressor=pipeline_compressor,
    base_retriever=base_retriever
)

In [ ]:
### 14. 검색
question = "미국 대선 후보들의 경제 정책은?"
docs = post_retriever.invoke(question)

In [ ]:
### 15. context
context = "\n\n".join(
    doc.page_content
    for doc in docs
)

In [ ]:
### 16. prompt
prompt = ChatPromptTemplate.from_template(
"""
You are a helpful AI assistant.

Use ONLY the following context.

Context:
{context}

Question:
{input}

Answer:
"""
)

In [ ]:
### 17. Chain
chain = prompt | model | StrOutputParser()

In [ ]:
### 18. 실행
result = chain.invoke(
  {
      "context":context,
      "question":question
  }
)
print(result)

제공된 문서에 따르면, 미국 대선 후보들의 경제 정책은 다음과 같은 특징을 보입니다.

*   **트럼프 대통령:** 숫자(흑자)에 매우 민감하며, 당장 흑자를 줄이라는 요구를 할 가능성이 높습니다.
*   **할리 해리스 후보:** 바이든 대통령의 입장을 거의 계승할 것으로 예상됩니다.
*   **전반적인 경향:** 누가 대통령이 되든 미국 경제는 국내 기업에 부담을 가중시킬 수 있습니다.
*   **정책 방향:** 전반적으로 미국의 대외 경제 정책은 과거 트랜스파던트 시대의 자유무역 촉진에서 벗어나 보호주의 및 보호 무역을 강조하는 방향으로 전환될 것으로 예상됩니다.

이 정보는 제공된 문서에 기반하며, 실제 후보들의 구체적인 경제 정책과는 차이가 있을 수 있습니다.
